In [1]:
import os
import re
import optional
import requests
import json
import textwrap
from pydantic import BaseModel, Field, TypeAdapter, ValidationError
from typing import List
import torch
from pathlib import Path
import pandas as pd
from pypdf import PdfReader
from deep_translator import GoogleTranslator
from langdetect import detect


In [2]:
API_URL = "http://localhost:8000/generate"


In [3]:
print(torch.cuda.is_available())

True


In [4]:
pdf_path = "/home/sajin/keymetrics_sajin/keymetrics_sajin/policy_docs/policy_docs/margade-metsaelupaigatuupide-tegevuskava.pdf"

In [5]:
#adaptation based on previous metrics extracted.
#more metric hits to be added with upcoming metrics extracted & manual annotation of policy documents.
METRIC_HINTS = [
    r"\b\d+(\.\d+)?\b",
    r"\bpercent\b|\b%\b",
    r"\bhectares?\b|\bha\b",
    r"\btonnes?\b|\btCO2e\b|\bCO2\b",
    r"\b€\b|\beuro\b|\bmillion\b|\bbillion\b",
    r"\btarget\b|\brestore\b|\breduce\b|\bincrease\b|\bachieve\b",
    r"\bby 20\d{2}\b"
]

def is_metric_candidate(text):
    return any(re.search(pattern, text, re.IGNORECASE) for pattern in METRIC_HINTS)

In [6]:
def read_pdf_file(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""

        if text.strip():
            pages.append({
                "page": i + 1,
                "text": text
            })

    return pages

In [7]:
def chunk_pdf_pages(pages, chunk_size=250, overlap=30):
    chunks = []

    for page in pages:
        text = page["text"]
        start = 0

        while start < len(text):
            end = start + chunk_size

            chunks.append({
                "page": page["page"],
                "text": text[start:end]
            })

            start = end - overlap

    return chunks

In [8]:
# translator approach might be changed based on yasar's suggestions. Any suggestions on this are welcome, please.
def translate_to_english(text):

    if not text or len(text.strip()) < 5:
        return text, "unknown"

    try:
        detected_lang = detect(text)
    except:
        detected_lang = "unknown"


    if detected_lang == "en":
        return text, detected_lang

    try:
        translated_text = GoogleTranslator(
            source='auto',
            target='en'
        ).translate(text)

        return translated_text, detected_lang

    except Exception as e:
        print("Translation failed:", e)
        return text, detected_lang


In [9]:
def extract_from_pdf(pdf_path):

    pages = read_pdf_file(pdf_path)
    chunks = chunk_pdf_pages(pages)

    all_metrics = []

    for i, chunk in enumerate(chunks):
        
        if not is_metric_candidate(chunk["text"]):
            continue

        print(f"\nProcessing chunk {i+1}/{len(chunks)} | page {chunk['page']}")

        original_text = chunk["text"]

    
        translated_text, detected_lang = translate_to_english(original_text)

        print(f"Detected language: {detected_lang}")

    
        prompt = build_prompt(translated_text)

        raw_output = call_model(prompt)

        metrics = parse_output(
            raw_output,
            page_number=chunk["page"]
        )
        
        
        
        for m in metrics:
            m.page = chunk["page"]
            m.chunk_id = i
            m.original_text = original_text
            m.translated_text = translated_text
            m.detected_language = detected_lang

        all_metrics.extend(metrics)

    return all_metrics

In [10]:
class ExtMetric(BaseModel):
    extracted_text: str = Field(description="The word or phrase from the text")
    metric_level: str = Field(description="Action, Outcome, or Unsure")
    metric_class: str = Field(description="Area, Emissions, Spending, etc.")
    original_text: str = Field(default="")
    translated_text: str = Field(default="")
    detected_language: str = Field(default="")
    chunk_id: int | None = None
    page: int | None = None

In [ ]:
def build_prompt(text):
    return f"""
You are extracting key metrics from peatland-related policy documents.

A key metric must be:
1. Quantifiable
2. Connected to peatland, wetland, land-use, restoration, conservation, drainage, emissions, biodiversity, or policy implementation
3. Useful for measuring policy action, outcome, target, funding, area, emissions, or progress

Do NOT extract:
- document titles
- section headings
- generic policy names
- dates alone
- page numbers
- references
- numbers unrelated to policy action or outcome
- vague statements without measurable value

Return ONLY valid JSON.
Return an empty list [] if there are no valid metrics.

JSON schema:
[
  {{
    "extracted_text": "exact metric phrase from text",
    "metric_level": "Action | Outcome | Target | Input | Context",
    "metric_class": "Area | Emissions | Funding | Count | Percentage | Timeline | Biodiversity | Other",
    "unit": "ha | tonnes CO2e | % | EUR | years | count | unknown",
    "policy_relevance": "brief reason why this is a policy metric"
  }}
]

Text:
\"\"\"{text}\"\"\"
"""

In [12]:
def call_model(prompt: str):
    payload = {
        "prompt": prompt,
        "max_new_tokens": 1024,
        "temperature": 0.1,
        "top_p": 0.9   
    }

    response = requests.post(API_URL, json=payload)
    
    print("STATUS CODE:", response.status_code)
    print("RAW SERVER RESPONSE:", response.text)

    if response.status_code != 200:
        print("SERVER ERROR:")
        print(response.text)
        return ""

    data = response.json()

    if "response" not in data:
        print("SERVER ERROR RESPONSE:", data)
        return ""

    raw_text = data["response"]

    print("OUTPUT METRICS:\n", raw_text)

    return raw_text.strip()

In [ ]:
#extract JSON block

In [38]:
"""def parse_output(raw_text: str):
    try:
        
        start = raw_text.find("[")
        end = raw_text.rfind("]") + 1
        json_str = raw_text[start:end]

        data = json.loads(json_str)

        return [ExtMetric(**item) for item in data]

    except Exception as e:
        print("Parsing failed:", e)
        print("Raw output:", raw_text)
        return []
"""

'def parse_output(raw_text: str):\n    try:\n\n        start = raw_text.find("[")\n        end = raw_text.rfind("]") + 1\n        json_str = raw_text[start:end]\n\n        data = json.loads(json_str)\n\n        return [ExtMetric(**item) for item in data]\n\n    except Exception as e:\n        print("Parsing failed:", e)\n        print("Raw output:", raw_text)\n        return []\n'

In [13]:
def parse_output(raw, page_number=None):
    import json
    import re

    try:
        data = json.loads(raw)

        if isinstance(data, dict):
            data = [data]

    except Exception:
        matches = re.findall(r'\{.*?\}', raw, re.DOTALL)
        data = []

        for match in matches:
            try:
                obj = json.loads(match)
                data.append(obj)
            except Exception:
                pass

    cleaned = []

    valid_levels = [
        "Action",
        "Outcome",
        "Unsure"
    ]

    valid_classes = [
        "Area",
        "Emissions",
        "Site Status",
        "Spending",
        "Policy Action",
        "Knowledge Resource",
        "Practical Resource",
        "Environment Quality",
        "Miscellaneous"
    ]

    bad_phrases = [
        "student enrollment",
        "student population",
        "temperature of the site",
        "generic infrastructure",
        "technical specifications"
    ]
    
    EXAMPLE_LEAKS = [
        "12,000 hectares",
        "€50 million",
        "2.4 million tCO2e",
        "450 plastic piling dams"
    ]

    for item in data:

        extracted_text = item.get("extracted_text", "").strip()
        metric_level = item.get("metric_level", "Unsure").strip()
        metric_class = item.get("metric_class", "Miscellaneous").strip()

        
        if metric_level not in valid_levels:
            metric_level = "Unsure"

        if metric_class not in valid_classes:
            metric_class = "Miscellaneous"

        cleaned.append(
            ExtMetric(
                extracted_text=extracted_text,
                metric_level=metric_level,
                metric_class=metric_class,
                page=page_number
            )
        )

    return cleaned

In [ ]:
"""def parse_output(raw):
    try:
        # extract JSON block 
        start = raw.find("[")
        end = raw.rfind("]") + 1
        clean = raw[start:end]

        return json.loads(clean)
    except Exception as e:
        print("Parsing failed:", e)
        print("Raw output:", raw)
        return []
"""

In [ ]:
"""gen_test_text = """
Peatland Restoration and Climate Action Policy 2026
By 2030, the government commits to the rewetting of 50,000 hectares.
We anticipate a reduction of 4.2 million tCO2e by 2040.
A budget of €120 million has been allocated.
"""


In [127]:
gen_test_text = textwrap.dedent("""
Peatland Restoration and Climate Action Policy 2026
1. Land Management and Conservation Targets
The Department aims to enhance the resilience of national carbon sinks through active intervention.
By 2030, the government commits to the rewetting of 50,000 hectares of degraded raised bogs.
This target focuses on the SAC-designated North-Western Complex, which has been prioritized for immediate hydrological restoration.
Conversely, the urbanization of suburban zones has increased by 12% in the last decade, a trend that must be decoupled from environmental planning.
2. Emissions Reductions and Air Quality
A primary objective of this framework is the mitigation of greenhouse gas release.
We anticipate a total reduction of 4.2 million tCO2e by 2040 specifically from peatland re-vegetation.
This is distinct from our national goal to reduce passenger vehicle emissions by 15%, which is handled under the Transport Directive.
Furthermore, the Peatland Carbon Assessment Report will be published annually to track progress.
3. Financial Allocation and Scheme Development
To support rural communities, the Minister has authorized the Peatland Agri-Environmental Scheme (PAES) to provide direct support to landowners.
A dedicated budget of €120 million has been ring-fenced for this purpose.
In comparison, the Department of Education has noted that student enrollment has reached 600,000 pupils, requiring a separate infrastructure budget not covered under this environmental mandate.
4. Technical Specifications and Restoration Depth
Restoration success is measured by the stability of the water table.
We require the installation of 450 plastic piling dams per site to ensure water levels remain within 10cm of the surface.
During the pilot phase, the average temperature of the site was recorded at 18°C, which, while noted by researchers, is not a metric for policy delivery or outcome.
""")

In [ ]:
"""prompt = build_prompt(extract_from_pdf)

raw_output = call_model(prompt)

print("\nRAW OUTPUT:\n")
print(raw_output)

metrics = parse_output(raw_output, page_number=chunk["page"])

for m in metrics:
    

    print("\nPARSED METRICS:\n")

if not metrics:
    print("No metrics extracted.")

else:
    for m in metrics:
        print(
            f"{m.extracted_text} | "
            f"{m.metric_level} | "
            f"{m.metric_class}"
            f"confidence={m.confidence}"
        )
        """

In [14]:
def save_metrics(metrics, output_csv="pdf_metrics.csv", output_json="pdf_metrics.json"):
    rows = [m.model_dump() for m in metrics]

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(metrics)} metrics")

In [15]:
metrics = extract_from_pdf(pdf_path)

for m in metrics:
    print(
        f"Page {m.page} | "
        f"{m.extracted_text} | "
        f"{m.metric_level} | "
        f"{m.metric_class} | "
        f"Detected Language: {m.detected_language}"
    )

save_metrics(metrics)


Processing chunk 1/1163 | page 1
Detected language: et
STATUS CODE: 200
RAW SERVER RESPONSE: {"response":"user\n[]"}
OUTPUT METRICS:
 user
[]

Processing chunk 3/1163 | page 2
Detected language: so
STATUS CODE: 200
RAW SERVER RESPONSE: {"response":"user\n\nBased on the provided text, there are no valid metrics that meet the criteria. The text does not contain any quantifiable information related to peatland, wetland, land-use, restoration, conservation, drainage, emissions, biodiversity, or policy implementation.\n\nHere is the empty JSON response:\n\n```json\n[]\n```\n\nIf you have more text to analyze, please provide it."}
OUTPUT METRICS:
 user

Based on the provided text, there are no valid metrics that meet the criteria. The text does not contain any quantifiable information related to peatland, wetland, land-use, restoration, conservation, drainage, emissions, biodiversity, or policy implementation.

Here is the empty JSON response:

```json
[]
```

If you have more text to analy